# 실험 2 — 1. 데이터 준비 (다운사이징 · 변수 제거 · 기상 창 집계 · 딥러닝용 시계열 배열)

실험 1(`EXP/1/3_기상조인_후_학습.ipynb`)이 만든 데이터(LIMS 칼럼 + 수분활성도·식품pH·염도 + 기상·대기 17변수 × D-1~D-100)에서 출발해, 실험 2 의 모든 모델(`2_autogluon.ipynb`, `3_딥러닝.ipynb`)이 같은 입력을 쓰도록 파일로 저장한다. 결정 근거는 `docs/04_실험계획/성능향상_방안.md`.

| 단계 | 내용 | 근거 |
|---|---|---|
| 다운사이징 | **전체 기간(train·val·test)에 검출이 한 건도 없는 품목명(`train_prdlst_nm`)의 행을 모두 삭제**(38종·76,497행 → 600,040행). 품목 목록을 정할 때 val·test 의 검출 여부도 본다 | 사용자 지시 2026-10-01 (2026-09-29 의 "train 기간만 보고 정한 37종"에서 바꿈) |
| 변수 제거 1 | `lat`·`lon`·`legal_dong_code` 원값 — **2026-10-01 `EXP/1/1_데이터정제.ipynb` 로 옮김**(입력 `X.parquet` 에 이미 없다) | 성능향상_방안 §3-1 (장소를 외워 test 에서 무너짐) |
| 변수 제거 2 | 기상 하루값 1,700칼럼 → **창 집계**(D-1~7·14·30·100 의 평균·최고·최저·표준편차, 강수 합·강수일·최장 무강수, 생육 범위 일수, 결측 일수)로 대체 | §3-2, §2-7, §2-11 |
| 변수 제거 3 | `train_prdlst_nm` (`train_prdlst_code` 와 1:1) — **2026-10-01 `EXP/1/1_데이터정제.ipynb` 로 옮김**(입력 `X.parquet` 에 이미 없다) | §3-3 |
| 변수 제거 4 | `jdgmnt_fom_code` — 실험 1 입력에 없어 할 일 없음 | §3-4 |
| 딥러닝 입력 | 논문(`docs/참고자료/0808Manuscript.2.docx`)의 시계열 가지(Conv1D–BiLSTM–Attention)는 **하루값 순서열**이 필요하므로, 창 집계와 별도로 17변수 × 100일 배열을 표준화해 저장한다(트리 모델에는 넣지 않음) | 논문 Table 2 |

**산출물** (`data/exp2/`, git 제외)

| 파일 | 내용 |
|---|---|
| `tabular.parquet` | 행 = 다운사이징 뒤 시료. `lims_data_key`, `split`(train/val/test), `y`, 범주형 8개(LIMS 5 + 식품 값 표기 3), 식품 3개, 기상 창 집계. 값은 원자료 그대로(결측 유지, 표준화 안 함) |
| `seq_std.npy` | (행 수, 100일, 17변수) float16. 시간 순서(0 = D-100 … 99 = D-1). **train 행의 평균·표준편차로 표준화**하고 결측은 0(= 평균) |
| `meta.json` | 삭제한 품목 목록, 칼럼 목록, 표준화 통계, 행 수 |

기준일 당일(D0)은 실험 1과 같이 넣지 않는다. 모든 통계(표준화)는 train 행으로만 계산한다.

**2026-10-01 변경**: 입력 `X.parquet` 에서 검사기관 코드 `inspct_instt_code` 가 빠졌고(예측 시점에 알 수 없는 값), 로마자로 적힌 한국 주소 78행이 `src_route` domestic 으로 옮겨져 국내 관측소 기상이 붙었다(`EXP/1/1_데이터정제.ipynb`·`3_기상조인_후_학습.ipynb`). 식품 값이 원문에서 어떤 표기였는지(단일값·중앙·이상·이하·초과·미만)를 담은 `aw_label`·`ph_label`·`na_label` 을 범주형 피처로 더했다.

**2026-10-01 변경 (다운사이징 기준)**: 삭제 기준을 "train 기간 검출 0 인 `train_prdlst_code` 37종(76,462행)"에서 "전체 기간 검출 0 인 `train_prdlst_nm` 38종(76,497행)"으로 바꿨다(사용자 지시). 더 지워지는 것은 train 에 행이 없는 `환경검체`(`Z4100000000000`, val 21·test 14행, 검출 0) 35행이고, 남는 행은 600,075 → 600,040 이다.

**2026-10-01 변경 (변수 제거 1·3 을 실험 1 로 옮김)**: `lat`·`lon`·`legal_dong_code`·`train_prdlst_nm` 제거를 `EXP/1/1_데이터정제.ipynb` 로 옮겼다(사용자 지시). 같은 노트북에서 `train_prdlst_code` 결측 170행을 `'기타'`(그 행의 `train_prdlst_nm` 값)로 채운다. 그래서 이 노트북의 다운사이징은 품목명 `train_prdlst_nm` 대신 품목코드 `train_prdlst_code` 로 묶는다 — 둘은 1:1 이라 지워지는 행은 같다. 표에 보이는 품목명은 원자료 `data/listeria_body.parquet` 에서 코드별로 가져온다.

## 1. 설정

In [1]:
import gc
import json
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path(r'C:\Users\NT751\Desktop\천영진\리스테리아ML')
X_path = PROJECT_DIR / 'data' / 'X.parquet'
y_path = PROJECT_DIR / 'data' / 'y.parquet'
JOIN_DIR = PROJECT_DIR / 'data' / 'collected' / 'exp1_wx'   # 실험 1 의 기상·식품 조인 결과
OUT_DIR = PROJECT_DIR / 'data' / 'exp2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

KEY_COL, DATE_COL = 'lims_data_key', 'mnfctur_de'
TRAIN_END = '2024-06-30'
VAL_START, VAL_END = '2024-07-01', '2025-06-30'
TEST_START, TEST_END = '2025-07-01', '2026-06-30'

BODY_PATH = PROJECT_DIR / 'data' / 'listeria_body.parquet'   # 품목명을 표에 보여 줄 때만 읽는다
ITEM_COL = 'train_prdlst_code'   # 다운사이징 기준 품목 칼럼 (학습용 품목코드. train_prdlst_nm 과 1:1 이고 코드가 비어 있던 170행은 '기타')
DROP_COLS = []   # lat·lon·legal_dong_code·train_prdlst_nm 은 2026-10-01 부터 EXP/1/1_데이터정제.ipynb 에서 뺀다 (성능향상_방안 §3-1·§3-3)
DROPPED_UPSTREAM = ['lat', 'lon', 'legal_dong_code', 'train_prdlst_nm']
FOOD_COLS = ['aw', 'ph', 'na']
LAG_DAYS = 100
WINDOWS = [7, 14, 30, 100]

# 실험 1 과 같은 17변수 (칼럼 이름, 한글 이름, 단위)
VARIABLES = [
    ('avg_temp', '평균기온', '°C'), ('max_temp', '최고기온', '°C'), ('min_temp', '최저기온', '°C'),
    ('rain', '강수량', 'mm'), ('avg_wind', '평균풍속', 'm/s'), ('max_wind', '최대풍속', 'm/s'),
    ('sunshine', '일조시간', '시간'), ('solar_rad', '일사량', 'MJ/m²'),
    ('soil_t5', '지중온도 5cm', '°C'), ('soil_t10', '지중온도 10cm', '°C'), ('soil_t20', '지중온도 20cm', '°C'),
    ('soil_t30', '지중온도 30cm', '°C'), ('soil_t_nas', '지중온도(NAS, 깊이 정의서 설명 없음)', '°C'),
    ('soil_moist', '토양수분', '%'), ('no2', '이산화질소', 'ppm'), ('co', '일산화탄소', 'ppm'), ('so2', '아황산가스', 'ppm'),
]
WX_VARS = [v[0] for v in VARIABLES]
WX_KO = {v[0]: v[1] for v in VARIABLES}

# 생육 범위 (리서치팀 예측인자 PPT — 성능향상_방안 §2-11): 기온 최적 30~37℃ · 한계 −2~45℃, 토양온도 최적 8~15℃ · 한계 8~30℃
TEMP_OPT = (30.0, 37.0); TEMP_LIM = (-2.0, 45.0)
SOIL_OPT = (8.0, 15.0);  SOIL_LIM = (8.0, 30.0)
SOIL_T_VARS = ['soil_t5', 'soil_t10', 'soil_t20', 'soil_t30', 'soil_t_nas']
print('출력 폴더:', OUT_DIR)

출력 폴더: C:\Users\NT751\Desktop\천영진\리스테리아ML\data\exp2


## 2. 실험 1 데이터 불러오기

`X.parquet`(7열 = 행 키 + 기준일 + LIMS 범주형 5개)·`y.parquet`(행 키 + 타깃)·`exp1_wx/food.parquet`(수분활성도·식품pH·염도)를 `lims_data_key` 로 맞춘다. 기상 파일은 변수마다 따로 읽는다(6절).

In [2]:
X = pd.read_parquet(X_path)
y_df = pd.read_parquet(y_path)
assert X[KEY_COL].is_unique and y_df[KEY_COL].is_unique
y = y_df.set_index(KEY_COL)['judge_result'].reindex(X[KEY_COL]).set_axis(X.index)
assert y.notna().all()
FOOD_LABEL_COLS = [f'{c_}_label' for c_ in FOOD_COLS]   # 식품 값 표기: 단일값·중앙·이상·이하·초과·미만 (값 없으면 '')
FOOD_LABELS = ['단일값', '중앙', '이상', '이하', '초과', '미만']
food = pd.read_parquet(JOIN_DIR / 'food.parquet', columns=[KEY_COL] + FOOD_COLS + FOOD_LABEL_COLS)
assert (food[KEY_COL].to_numpy() == X[KEY_COL].astype(str).to_numpy()).all(), '식품 파일 행 순서가 X 와 다름'
for v in WX_VARS:   # 기상 파일 행 순서 확인
    keys = pq.read_table(JOIN_DIR / f'wx_{v}.parquet', columns=[KEY_COL]).column(0).to_numpy()
    assert (keys == X[KEY_COL].astype(str).to_numpy()).all(), v
print('X', X.shape, '| 검출', int(y.sum()), '| 식품', food.shape, '| 기상 파일 17개 행 순서 일치')

X (676537, 11) | 검출 1969 | 식품 (676537, 7) | 기상 파일 17개 행 순서 일치


## 3. 시간순 분할 (실험 1과 같음)

train ~2024-06-30 · val 2024-07-01~2025-06-30 · test 2025-07-01~2026-06-30. 이후 "train 기준" 계산(표준화 통계)은 이 train 행만 쓴다. 다운사이징 품목 목록만 전체 기간으로 정한다(4절, 2026-10-01 변경).

In [3]:
d = X[DATE_COL]
split = pd.Series(np.where(d <= pd.Timestamp(TRAIN_END), 'train',
                  np.where(d.between(pd.Timestamp(VAL_START), pd.Timestamp(VAL_END)), 'val',
                  np.where(d.between(pd.Timestamp(TEST_START), pd.Timestamp(TEST_END)), 'test', ''))), index=X.index)
assert (split != '').all(), '어느 구간에도 속하지 않는 행이 있음'
def split_table(mask=None):
    m = pd.Series(True, index=X.index) if mask is None else mask
    return pd.DataFrame([{'구간': s, '행 수': int((m & (split == s)).sum()), '검출': int(y[m & (split == s)].sum()),
                          '검출 비율': round(float(y[m & (split == s)].mean()), 5)} for s in ['train', 'val', 'test']])
display(split_table())

,구간,행 수,검출,검출 비율
0,train,497896,1491,0.00299
1,val,87553,191,0.00218
2,test,91088,287,0.00315


## 4. 다운사이징 — 전체 기간에 검출이 없는 품목 삭제

- 품목 = `train_prdlst_code`(모델팀 학습용 품목코드, 53종 = 코드 52종 + `'기타'`). `train_prdlst_nm` 과 1:1 이고, 원자료에서 코드가 비어 있던 170행은 `EXP/1/1_데이터정제.ipynb` 에서 `'기타'` 로 채워져 한 그룹이다. 품목명은 `X.parquet` 에 없어 원자료 `data/listeria_body.parquet` 에서 코드별로 가져와 표에만 보여 준다.
- **전체 기간(train·val·test)에서 검출 0 인 품목**을 고르고, 그 품목의 행을 train·val·test 모두에서 지운다(사용자 지시 2026-10-01). 2026-09-29~10-01 에는 train 행만 보고 골랐다(37종·76,462행). 전체 기간 기준은 38종·76,497행이고, 차이는 train 에 행이 없는 품목 `환경검체`(`Z4100000000000`, val 21·test 14행, 검출 0) 하나다.
- 이 기준은 품목 목록을 정할 때 val·test 의 정답(검출 여부)을 본다. 지운 품목은 전체 기간 검출이 0 이므로 val·test 검출 건수는 줄지 않는다.


In [4]:
item = X[ITEM_COL].astype(str)                       # 품목코드 (결측 없음 — 코드가 비어 있던 행은 '기타')
assert X[ITEM_COL].notna().all()
assert not set(DROPPED_UPSTREAM) & set(X.columns), 'X.parquet 에 1_데이터정제에서 빼기로 한 칼럼이 남아 있음 — 1_데이터정제를 다시 실행'
# 품목명은 X.parquet 에 없다 — 표에 보여 주려고 원자료에서 (코드, 이름) 쌍만 가져온다 (코드가 빈 행은 '기타')
_pair = pd.read_parquet(BODY_PATH, columns=['train_prdlst_code', 'train_prdlst_nm']).astype('string').fillna('기타').drop_duplicates()
assert _pair['train_prdlst_code'].is_unique and _pair['train_prdlst_nm'].is_unique, '품목코드와 품목명이 1:1 이 아님'
ITEM_NM = dict(zip(_pair['train_prdlst_code'], _pair['train_prdlst_nm']))
g = pd.DataFrame({'item': item, 'y': y, 'split': split})
al = g.groupby('item')['y'].agg(['size', 'sum'])      # 전체 기간 (train·val·test)
ZERO_CODES = sorted(al.index[al['sum'] == 0])         # 전체 기간 검출 0 품목코드
ZERO_ITEMS = [ITEM_NM[c_] for c_ in ZERO_CODES]       # 같은 품목의 이름 (train_prdlst_nm)
drop_mask = item.isin(ZERO_CODES)

sp = pd.crosstab(item, split).reindex(columns=['train', 'val', 'test'], fill_value=0)
zero_tab = (al.loc[ZERO_CODES].rename(columns={'size': '전체행', 'sum': '전체검출'})
            .assign(품목명=ZERO_ITEMS,
                    train행=sp.loc[ZERO_CODES, 'train'], val행=sp.loc[ZERO_CODES, 'val'], test행=sp.loc[ZERO_CODES, 'test'])
            .sort_values('전체행', ascending=False))
print(f'전체 기간 검출 0 품목 {len(ZERO_ITEMS)}종 → 삭제 행 {int(drop_mask.sum()):,} / 남는 행 {int((~drop_mask).sum()):,}')
print('삭제 품목의 검출 건수:', int(zero_tab['전체검출'].sum()), '(전체 기간 검출 0 인 품목만 지우므로 0)')
print('삭제 품목 중 train 에 행이 없는 품목:', zero_tab.loc[zero_tab['train행'] == 0, '품목명'].tolist())
display(zero_tab)
print('남는 행 구간별:')
display(split_table(~drop_mask))

KEEP = np.nonzero((~drop_mask).to_numpy())[0]        # 남는 행 번호 (X 순서)
Xk = X.iloc[KEEP].reset_index(drop=True)
yk = y.iloc[KEEP].reset_index(drop=True).astype('int8')
splitk = split.iloc[KEEP].reset_index(drop=True)
foodk = food.iloc[KEEP].reset_index(drop=True)
N = len(Xk)
IS_TRAIN = (splitk == 'train').to_numpy()


전체 기간 검출 0 품목 38종 → 삭제 행 76,497 / 남는 행 600,040
삭제 품목의 검출 건수: 0 (전체 기간 검출 0 인 품목만 지우므로 0)
삭제 품목 중 train 에 행이 없는 품목: ['환경검체']


,전체행,전체검출,품목코드,train행,val행,test행
item,,,,,,
알가공품류,29392,0,C0317000000000,21903,3737,3752
빙과류,22206,0,C0302000000000,16602,2826,2778
조리식품 등,9590,0,G0100000000000,6568,1285,1737
조미식품,4165,0,C0312000000000,3709,224,232
절임류 또는 조림류,2804,0,C0313000000000,2479,179,146
특수영양식품,2140,0,C0310000000000,1781,207,152
채소류,904,0,A0600000000000,573,114,217
코코아가공품류 또는 초콜릿류,735,0,C0303000000000,680,24,31
과일류,702,0,A0500000000000,633,29,40


남는 행 구간별:


,구간,행 수,검출,검출 비율
0,train,440116,1491,0.00339
1,val,78420,191,0.00244
2,test,81504,287,0.00352


## 5. 변수 제거

**2026-10-01**: 아래 `lat`·`lon`·`legal_dong_code`·`train_prdlst_nm` 은 `EXP/1/1_데이터정제.ipynb` 에서 이미 빠져 `X.parquet` 에 없다. 이 절은 남은 칼럼을 범주형 피처로 정리하고 식품 값 표기를 더한다.

- `lat`·`lon`·`legal_dong_code`: 좌표·법정동 원값. 실험 1 에서 LightGBM 이득의 65%/35% 가 `lat`/`lon` 이었고 빼면 test roc_auc 가 오르는 것을 확인했다(성능향상_방안 §0 진단 1).
- `train_prdlst_nm`: `train_prdlst_code` 와 1:1 이라 중복.
- `jdgmnt_fom_code`: 실험 1 입력(`X.parquet`)에 없다. 실험 2 도 LIMS 원본을 새로 붙이지 않으므로 해당 없음.
- 남는 LIMS 범주형 5개: `src_route`, `prdlst_code`, `train_prdlst_code`, `date_basis`, `addr_source`. `inspct_instt_code`(검사기관)는 2026-10-01 에 `X.parquet` 에서 제거됐다(예측 시점에 알 수 없는 값 — 교수님 지적, 사용자 결정).
- 식품 값 표기 3개(`aw_label`·`ph_label`·`na_label`)를 범주형으로 더한다 → 범주형 8개.

In [5]:
CAT_COLS = [c for c in Xk.columns if c not in (KEY_COL, DATE_COL) and c not in DROP_COLS]
assert all(str(Xk[c].dtype) == 'category' for c in CAT_COLS), '범주형이 아닌 칼럼이 남아 있음'
tab = pd.DataFrame({KEY_COL: Xk[KEY_COL].astype(str).to_numpy(), 'split': splitk.to_numpy(), 'y': yk.to_numpy(),
                    DATE_COL: Xk[DATE_COL].to_numpy()})
for c_ in CAT_COLS:
    tab[c_] = Xk[c_].cat.remove_unused_categories().to_numpy()   # 지운 품목의 범주값은 목록에서 뺀다
for c_ in FOOD_LABEL_COLS:   # 식품 값 표기 — 값이 없는 행은 결측
    tab[c_] = pd.Categorical(foodk[c_].replace('', np.nan).to_numpy(), categories=FOOD_LABELS)
CAT_COLS = CAT_COLS + FOOD_LABEL_COLS
for c_ in FOOD_COLS:
    tab[c_] = foodk[c_].astype('float32').to_numpy()
print('1_데이터정제에서 이미 제거:', DROPPED_UPSTREAM, '| 범주형 유지:', CAT_COLS, '| 식품:', FOOD_COLS)
print({c_: int(tab[c_].nunique()) for c_ in CAT_COLS})

제거: ['lat', 'lon', 'legal_dong_code', 'train_prdlst_nm'] | 범주형 유지: ['src_route', 'prdlst_code', 'train_prdlst_code', 'date_basis', 'addr_source', 'aw_label', 'ph_label', 'na_label'] | 식품: ['aw', 'ph', 'na']
{'src_route': 2, 'prdlst_code': 818, 'train_prdlst_code': 15, 'date_basis': 3, 'addr_source': 4, 'aw_label': 5, 'ph_label': 2, 'na_label': 2}


## 6. 기상 창 집계 (하루값 1,700칼럼 대체)

변수 17개마다, 기준일 전날부터 거슬러 올라가는 창 **7·14·30·100일**(D-1~D-w)에 대해:

| 집계 | 칼럼 이름 | 대상 |
|---|---|---|
| 평균·최고·최저·표준편차 | `{변수}_mean{w}`, `_max{w}`, `_min{w}`, `_std{w}` | 17변수 전부 |
| 강수 합, 강수일 수(>0mm), 최장 연속 무강수 일수 | `rain_sum{w}`, `rain_wet{w}`, `rain_maxdry{w}` | 강수량 |
| 생육 범위 일수 — 평균기온 ≥30℃(최적 하한), 30~37℃(최적), 최저기온 ≥−2℃(한계 하한 위) | `avg_temp_ge30_{w}`, `avg_temp_opt_{w}`, `min_temp_gem2_{w}` | 기온 |
| 토양온도 최적(8~15℃)·한계(8~30℃) 안 일수 | `{토양온도}_opt_{w}`, `_lim_{w}` | 지중온도 5개 |
| 100일 중 결측 일수 | `{변수}_nmiss100` | 17변수 전부 |

- 값이 없는 날은 평균·최고·최저·표준편차에서 뺀다(창 전체가 결측이면 NaN). 일수 계산은 값이 있는 날만 센다. 최장 무강수는 결측을 0mm 로 본다(논문과 같은 처리).
- 해외 행은 12개 변수(풍속 2·일조·지중온도 5·토양수분·대기 3)가 원래 없어 그 집계가 NaN 이다(실험 1 과 같음). 트리 모델은 NaN 을 그대로 받고, 딥러닝은 `3_딥러닝.ipynb` 에서 0(= train 평균)으로 채운다.
- 표준화는 하지 않는다(원자료 값 그대로 저장).

In [6]:
def window_aggs(v, A):
    """A: (행, 100) float32, 열 0 = D-1 … 열 99 = D-100. 반환: {칼럼: 값 배열}"""
    out = {}
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)   # 창 전체 결측 → NaN 경고 숨김
        for w in WINDOWS:
            W = A[:, :w]
            out[f'{v}_mean{w}'] = np.nanmean(W, axis=1)
            out[f'{v}_max{w}'] = np.nanmax(W, axis=1)
            out[f'{v}_min{w}'] = np.nanmin(W, axis=1)
            out[f'{v}_std{w}'] = np.nanstd(W, axis=1)
            if v == 'rain':
                out[f'rain_sum{w}'] = np.nansum(W, axis=1); out[f'rain_sum{w}'][np.isnan(W).all(axis=1)] = np.nan
                out[f'rain_wet{w}'] = (W > 0).sum(axis=1).astype(np.float32)
                dry = ~(W > 0)                                   # 결측은 0mm 로 본다
                run = np.zeros(len(W), dtype=np.int32); best = np.zeros(len(W), dtype=np.int32)
                for t in range(w):
                    run = np.where(dry[:, t], run + 1, 0); best = np.maximum(best, run)
                out[f'rain_maxdry{w}'] = best.astype(np.float32)
            if v == 'avg_temp':
                out[f'avg_temp_ge30_{w}'] = (W >= TEMP_OPT[0]).sum(axis=1).astype(np.float32)
                out[f'avg_temp_opt_{w}'] = ((W >= TEMP_OPT[0]) & (W <= TEMP_OPT[1])).sum(axis=1).astype(np.float32)
            if v == 'min_temp':
                out[f'min_temp_gem2_{w}'] = (W >= TEMP_LIM[0]).sum(axis=1).astype(np.float32)
            if v in SOIL_T_VARS:
                out[f'{v}_opt_{w}'] = ((W >= SOIL_OPT[0]) & (W <= SOIL_OPT[1])).sum(axis=1).astype(np.float32)
                out[f'{v}_lim_{w}'] = ((W >= SOIL_LIM[0]) & (W <= SOIL_LIM[1])).sum(axis=1).astype(np.float32)
    out[f'{v}_nmiss100'] = np.isnan(A).sum(axis=1).astype(np.float32)
    return {k: val.astype(np.float32) for k, val in out.items()}

SEQ_PATH = OUT_DIR / 'seq_std.npy'
seq = np.lib.format.open_memmap(SEQ_PATH, mode='w+', dtype=np.float16, shape=(N, LAG_DAYS, len(WX_VARS)))
SEQ_STATS = {}
agg_frames = []
t0 = time.time()
for j, v in enumerate(WX_VARS):
    cols = [f'{v}_d{k:03d}' for k in range(1, LAG_DAYS + 1)]
    A = pq.read_table(JOIN_DIR / f'wx_{v}.parquet', columns=cols).to_pandas().to_numpy(np.float32)[KEEP]   # (N, 100) D-1 … D-100
    agg_frames.append(pd.DataFrame(window_aggs(v, A)))
    # 딥러닝용 순서열: train 행 통계로 표준화, 시간 순서(D-100 → D-1)로 뒤집고, 결측 = 0
    mu = float(np.nanmean(A[IS_TRAIN])); sd = float(np.nanstd(A[IS_TRAIN]))
    sd = sd if sd > 0 else 1.0
    Z = (A[:, ::-1] - mu) / sd
    Z[np.isnan(Z)] = 0.0
    seq[:, :, j] = Z.astype(np.float16)
    SEQ_STATS[v] = {'mean': mu, 'std': sd, 'nan_frac': float(np.isnan(A).mean())}
    del A, Z; gc.collect()
    print(f'{v:11s} 집계 {agg_frames[-1].shape[1]:2d}칼럼 · train 평균 {mu:9.3f} 표준편차 {sd:8.3f} · 결측 {SEQ_STATS[v]["nan_frac"]:.4f} · {time.time() - t0:.0f}초')
seq.flush(); del seq
WX_AGG = pd.concat(agg_frames, axis=1)
del agg_frames; gc.collect()
tab = pd.concat([tab, WX_AGG], axis=1)
AGG_COLS = WX_AGG.columns.tolist()
print(f'기상 창 집계 {len(AGG_COLS)}칼럼 · 시계열 배열 {SEQ_PATH.name} {(N, LAG_DAYS, len(WX_VARS))} float16 {SEQ_PATH.stat().st_size / 1e9:.2f}GB')

avg_temp    집계 25칼럼 · train 평균    12.861 표준편차    9.923 · 결측 0.0000 · 4초


max_temp    집계 17칼럼 · train 평균    18.291 표준편차    9.939 · 결측 0.0000 · 7초


min_temp    집계 21칼럼 · train 평균     8.125 표준편차   10.390 · 결측 0.0000 · 10초


rain        집계 29칼럼 · train 평균     3.478 표준편차   12.316 · 결측 0.0006 · 15초


avg_wind    집계 17칼럼 · train 평균     1.751 표준편차    0.832 · 결측 0.0728 · 18초


max_wind    집계 17칼럼 · train 평균     4.302 표준편차    1.418 · 결측 0.0728 · 20초


... (8 outputs left out)

no2         집계 17칼럼 · train 평균     0.017 표준편차    0.011 · 결측 0.0728 · 68초


co          집계 17칼럼 · train 평균     0.429 표준편차    0.150 · 결측 0.0728 · 75초


so2         집계 17칼럼 · train 평균     0.003 표준편차    0.001 · 결측 0.0728 · 82초


기상 창 집계 353칼럼 · 시계열 배열 seq_std.npy (600040, 100, 17) float16 2.04GB


## 7. 저장과 요약

`tabular.parquet`(트리 모델·딥러닝 표 형식 가지 공통 입력), `seq_std.npy`(딥러닝 시계열 가지), `meta.json`(칼럼·품목·통계). 실험 2 의 다른 노트북은 이 세 파일만 읽는다.

In [7]:
FEATURES = CAT_COLS + FOOD_COLS + AGG_COLS
tab.to_parquet(OUT_DIR / 'tabular.parquet', index=False)
meta = {
    'made_at': datetime.now().isoformat(timespec='seconds'),
    'source': {'X': str(X_path), 'X_mtime': datetime.fromtimestamp(X_path.stat().st_mtime).isoformat(timespec='seconds'),
               'y': str(y_path), 'join_dir': str(JOIN_DIR)},
    'n_rows_before': int(len(X)), 'n_rows': int(N),
    'split_dates': {'train_end': TRAIN_END, 'val': [VAL_START, VAL_END], 'test': [TEST_START, TEST_END]},
    'split_counts': split_table(~drop_mask).to_dict('records'),
    'downsizing': {'item_col': ITEM_COL, 'rule': '전체 기간(train·val·test) 검출 0 인 품목코드(train_prdlst_code)의 행 삭제',
                   'zero_items': ZERO_ITEMS, 'zero_item_codes': ZERO_CODES, 'n_dropped_rows': int(drop_mask.sum())},
    'dropped_cols': ['(기상 하루값 1,700칼럼 → 창 집계로 대체)'], 'dropped_in_exp1_cleaning': DROPPED_UPSTREAM,
    'cat_cols': CAT_COLS, 'food_cols': FOOD_COLS, 'agg_cols': AGG_COLS, 'features': FEATURES,
    'windows': WINDOWS, 'lag_days': LAG_DAYS, 'thresholds': {'temp_opt': TEMP_OPT, 'temp_lim': TEMP_LIM, 'soil_opt': SOIL_OPT, 'soil_lim': SOIL_LIM},
    'seq': {'file': 'seq_std.npy', 'shape': [int(N), LAG_DAYS, len(WX_VARS)], 'dtype': 'float16', 'vars': WX_VARS,
            'time_axis': '0 = D-100 … 99 = D-1', 'standardized_with': 'train 행 평균·표준편차 (변수별)', 'nan_fill': 0.0, 'stats': SEQ_STATS},
}
(OUT_DIR / 'meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'행 {N:,} × 피처 {len(FEATURES)} (범주형 {len(CAT_COLS)} + 식품 {len(FOOD_COLS)} + 기상 집계 {len(AGG_COLS)})')
display(split_table(~drop_mask))
miss = tab[FOOD_COLS + AGG_COLS].isna().mean().sort_values(ascending=False)
print('결측 비율 상위 10칼럼 (해외 행에 없는 변수 = 약 7%):')
display(miss.head(10).round(4).to_frame('결측 비율').T)
for f in ['tabular.parquet', 'seq_std.npy', 'meta.json']:
    print(f'{f:18s} {(OUT_DIR / f).stat().st_size / 1e6:9,.1f} MB')

행 600,040 × 피처 364 (범주형 8 + 식품 3 + 기상 집계 353)


,구간,행 수,검출,검출 비율
0,train,440116,1491,0.00339
1,val,78420,191,0.00244
2,test,81504,287,0.00352


결측 비율 상위 10칼럼 (해외 행에 없는 변수 = 약 7%):


,ph,na,aw,so2_mean30,so2_max30,so2_min30,so2_std30,so2_mean100,so2_max100,so2_min100
결측 비율,0.6129,0.6038,0.5341,0.0728,0.0728,0.0728,0.0728,0.0728,0.0728,0.0728


tabular.parquet        337.2 MB
seq_std.npy          2,040.1 MB
meta.json                0.0 MB
